# Data engineering

By: Sherif Abdulkader Tawfik Abbas
[https://sheriftawfikabbas.github.io/](https://sheriftawfikabbas.github.io/)

Topics we will cover:
- Introduction to data engineering
- Common data engineering tasks (the "T" in ETL):
  - Data extraction
  - Turning text to lower-case
  - One-hot encoding
  - Removing outliers
  - Imputation
  - Data loading
- Number cruncher: the `numpy` library
  - `numpy` arrays
  - Array operations and broadcasting
  - `numpy` mathematical functions
  - Slicing, searching, sorting, filtering
- Data workhorse: `pandas`
  - The `DataFrame` object
  - Querying with `loc` and `iloc`
  - Loading and saving CSV files
  - Finding and filling missing values
  - One-hot encoding with `get_dummies`
  - Detecting and removing outliers
- Text manipulation in python

## Introduction to data engineering

Machine learning uses data to train a model to perform a predictive task. The data that machine learning models consume is different from the data a human can process. Humans can read text on a website and understand it using the rules of grammar plus common sense. A machine, on the other hand, can only process **numbers**. Data engineering is how we bridge that gap.

### Example 1: Numerical data

Let's say we want to pass global temperature data into a machine learning model, to train it to predict the global temperature in 20 years' time. The data looks like this:

```
%                  Monthly          Annual          Five-year        Ten-year        Twenty-year
% Year, Month,  Anomaly, Unc.,   Anomaly, Unc.,   Anomaly, Unc.,   Anomaly, Unc.,   Anomaly, Unc.

  1751    12    -2.169  3.365    -1.153    NaN       NaN    NaN       NaN    NaN       NaN    NaN
  1752     1    -3.587  3.193    -1.157  1.358       NaN    NaN       NaN    NaN       NaN    NaN
  1752     2     1.556  4.577    -1.208  1.381       NaN    NaN       NaN    NaN       NaN    NaN
  1752     9       NaN    NaN       NaN    NaN       NaN    NaN       NaN    NaN       NaN    NaN
```

The data can be obtained from this [link](http://berkeleyearth.lbl.gov/auto/Global/Complete_TAVG_complete.txt), and will be examined in Case Study 01 of this course. For every month — the *second* column, `Month`, after the `Year` — the temperature anomaly (the deviation from a reference average) is recorded. But some records contain `NaN`, which stands for "not a number". To pass the above table to a machine:

- First, we need to transform the above format into a format the machine can process, such as the CSV format.
- Second, we should substitute the `NaN` values with numerical values. This is called **data imputation**.

### Example 2: Natural language processing

> Foreign Affairs Minister Penny Wong has asked Optus to cover passport application fees for anyone caught up in last week's massive data breach, which affected millions of Australians.

How does the human brain comprehend this sentence? It applies rules of syntax, or grammar. The initial of Penny is upper-case, so we understand it's the name of a person. We know Optus is a company, not a person, so we won't be confused by "asked Optus". We know "caught up" means "involved". We have some idea about the security breach that hit the Optus servers. Finally, we know the sentence ended because of the full stop.

Is that how a machine would understand the text? The machine first needs to translate the text into a **numerical** form, and then process the numbers. As an example of a numerical form: representing every pair of consecutive words by a number:

| Two words | Number |
| --- | --- |
| has asked | 1 |
| caught up | 2 |
| which affected | 3 |

What if we also find "which affects", "have asked", and "catch up"? The meaning of "have asked" is the same as "has asked", so — given they have the same meaning — it makes more sense to represent them with the same number.

Therefore, before translating the words and sentences into numbers, we need to make things less confusing to the machine:
- plurals, such as "fees" and "millions", become singular
- different forms of a verb, such as "have" and "has", are represented as one verb, such as "have"
- we remove commas, full stops, etc., because they are not part of the vocabulary

The modifications to the data in the above two examples are known as **data preprocessing**, or more generally **data engineering**: the process of preparing data for machine learning. In python, it involves the use of several libraries, chiefly `numpy` and `pandas`.

## Common data engineering tasks

Data engineering is also known as **ETL**: extraction–transformation–loading. Below I will explain what each of those means.

### Data extraction

This is the process of getting the data from its source, in its *raw* form. For example: if we need stock prices to analyse, the extraction process includes writing scripts that collect the stock data from the stock-market platform.

The data, as-obtained, will be either poorly formatted or formatted in a way that is hard to analyse. For this reason we perform data **transformation** before analysis. Below are a few common transformation tasks.

### Turning text to lower-case

One of the most common transformations for text data is to unify the case: turning the text into lower-case. For example, if we collect user colour selections, users might enter `Red`, `BLUE`, `bLack`. Turning them all into lower-case makes the values consistent.

### One-hot encoding

When the data is categorical, such as the colours above, we should convert it into numerical form. Remember, in machine learning, the machine only understands numbers!

There are several possible ways to turn the colours into numbers:
1. Convert each colour into its three RGB values
2. Rank colours by integer numbers: red = 1, blue = 2, green = 3, black = 4, etc.
3. One-hot encode the colours

Option (1) makes sense here, since the RGB values genuinely describe what the colours are. Option (2) is not ideal in general, because most categories have no meaningful numerical order — a model would wrongly learn that "green is between blue and black". Option (3), **one-hot encoding**, is the most common approach: each category is represented by a vector of zeros with a single one at its own position:

```
red   -> [1, 0, 0, 0]
blue  -> [0, 1, 0, 0]
green -> [0, 0, 1, 0]
black -> [0, 0, 0, 1]
```

No category is "larger" than another, and the distance between any two categories is the same. We will implement this with `pandas` later in this class.

### Dealing with outliers

For numerical data, some values can be far lower or far higher than the majority of the data. These are called *outliers*, and they should be handled appropriately: extreme values can badly distort statistics such as the mean, and many models are sensitive to them. We will detect and remove outliers later in this class.

### Imputation

In addition to outliers, data can have missing or wrongly formatted values — *gaps* that you need to fill. Filling those gaps is known as **data imputation**. We will learn a few imputation techniques in this class.

### Data loading

Once we have transformed the data, we are ready to load it into data files, or databases, for further processing.

Another step that takes place during or after data transformation is *exploratory data analysis*, the topic of the [next class](../Class03_ExploratoryDataAnalysis/EDA.ipynb).

In this class, you will learn how to use `numpy` and `pandas`, as well as python string operations, to prepare your data.

## Number cruncher: the `numpy` library

`numpy` is one of the richest and most popular python libraries. You can browse its extensive collection of mathematical functions in the [numpy documentation](https://numpy.org/doc/stable/reference/index.html). Its popularity is built on one thing above all: the **numpy array**.

Why not just use python lists? Two reasons:

1. **Speed.** A list can hold anything — numbers, strings, other lists — so python checks the type of every element at every step. A numpy array holds numbers of one fixed type, packed contiguously in memory, and its operations are implemented in compiled C. On large arrays the difference is not a few per cent; it is tens to hundreds of times faster.
2. **Vectorisation.** With lists, operating on every element means writing a `for` loop. Numpy operations act on the whole array at once: `a * 4` multiplies every element, no loop needed.

### numpy arrays

Let's create an array of 3 numbers: 4, 6 and 7.

In [1]:
import numpy as np
a = np.array([4, 6, 7])
print(a)

[4 6 7]


Note the syntax: the `array` function expects a python collection, and here we pass a `list`.

Like python lists, we can find the number of elements with `len()`. Numpy arrays also have a `shape` **attribute** — a tuple giving the size of each dimension. (Watch out: `shape` is an attribute, not a function — writing `a.shape()` raises a `TypeError`.)

In [2]:
a = np.array([4, 6, 7])
print(len(a))
print(a.shape)

3
(3,)


This means we have a 1D array with 3 elements. Let's create a 2D array.

In [3]:
a = np.array([[4, 6, 7], [5, 6, 7]])
print(a)
print(a.shape)

[[4 6 7]
 [5 6 7]]
(2, 3)


To access the elements of a `numpy array`, we use the same bracket notation as with python lists. For example, to get the first row of the array above:

In [4]:
a = np.array([[4, 6, 7], [5, 6, 7]])
print(a[0])

[4 6 7]


### Array operations

We can multiply an array by a scalar:

In [5]:
a = np.array([[4, 6, 7], [5, 6, 7]])
print(a * 4)

[[16 24 28]
 [20 24 28]]


Behind the scenes, numpy applied the multiplication to *every* element — this is **vectorisation**, and the stretching of the single value `4` across all elements is called **broadcasting**. It also works between arrays of compatible shapes:

In [6]:
a = np.array([[1, 2, 3], [4, 5, 6]])
b = np.array([10, 20, 30])
print(a + b)   # b is broadcast across both rows

[[11 22 33]
 [14 25 36]]


Element-wise operations apply to all of python's arithmetic operators between two arrays of the same shape:

In [7]:
a = np.array([4, 6, 7])
b = np.array([7, 8, 9])
print(a * b)
print(a / b)
print(a + b)
print(a - b)

[28 48 63]
[0.57142857 0.75       0.77777778]
[11 14 16]
[-3 -2 -2]


What about matrix multiplication? For that we use the `dot()` function (or the `@` operator, which means the same thing):

In [8]:
a = np.array([[4, 6, 7], [0, 3, 2]])
b = np.array([[7, 9], [4, 10], [-4, 6]])
print(a.dot(b))
print(a @ b)   # equivalent modern syntax

[[ 24 138]
 [  4  42]]
[[ 24 138]
 [  4  42]]


### `numpy` mathematical functions

`numpy` has a huge number of mathematical functions. For an overview:
- [Statistical functions](https://numpy.org/doc/stable/reference/routines.statistics.html)
- [Mathematical functions](https://numpy.org/doc/stable/reference/routines.math.html) (trigonometry and much more)

Using them on an array is straightforward. Let's compute the sum, mean, maximum, minimum, standard deviation and variance of an array of numbers:

In [9]:
a = np.array([1, 5, 4, 7, 5, 3, 5.7, 3.4, 2, 6, 8, 7])
print(a.sum(), a.mean(), a.max(), a.min(), a.std(), a.var())

57.1 4.758333333333334 8.0 1.0 2.048356387177019 4.195763888888889


These are some of the common *descriptive statistics* we normally compute on numerical data before machine learning — we will meet them again in the EDA class.

We can also apply mathematical functions element-wise. Let's try rounding, floor and sine:

In [10]:
a = np.array([1.67, 5.8, 4.2, 9.7, 5, 3, 5.7, 3.4])
print(np.round(a))
print(np.floor(a))
print(np.sin(a))

[ 2.  6.  4. 10.  5.  3.  6.  3.]
[1. 5. 4. 9. 5. 3. 5. 3.]
[ 0.99508335 -0.46460218 -0.87157577 -0.27176063 -0.95892427  0.14112001
 -0.55068554 -0.2555411 ]


### Slicing numpy arrays

Like python lists and tuples, numpy arrays can be sliced with the `:` operator. Let's take the first 3 elements of a 6-element array into a new array:

In [11]:
a = np.array([1, 2, 3, 4, 5, 6])
b = a[0:3]
print(b)

[1 2 3]


We can also use negative indices (counting from the end) and a step value:

In [12]:
a = np.array([1, 2, 3, 4, 5, 6])
print(a[-1])      # the last element
print(a[0:4:2])   # the first 4 elements, in steps of 2

6
[1 3]


### Iterating through a numpy array

We can iterate through the elements of a numpy array just like a python list:

In [13]:
a = np.array([1, 2, 3, 4, 5, 6])
for i in a:
    print(i)

1
2
3
4
5
6


You can see we only get the elements, not their indices. To get both index and element in the loop, use the `ndenumerate()` function:

In [14]:
a = np.array([1, 2, 3, 4, 5, 6])
for index, i in np.ndenumerate(a):
    print(index, i)

(0,) 1
(1,) 2
(2,) 3
(3,) 4
(4,) 5
(5,) 6


### Concatenating arrays

We cannot join two numpy arrays with `+`, because `+` is reserved for element-wise addition. Instead we use the `concatenate()` function:

In [15]:
a = np.array([1, 2, 3, 4, 5, 6])
b = np.array([10, 11, 12])
c = np.concatenate([a, b])
print(c)

[ 1  2  3  4  5  6 10 11 12]


Note: we must place the arrays in a python list or tuple before passing them to `concatenate`.

### Searching an array

In a python list, `in` tells you *whether* an element exists, but finding *where* requires writing a loop. In numpy, the `where()` function gives you the indices directly:

In [16]:
a = np.array([1, 2, 3, 4, 5, 6])
print(np.where(a == 3))

a = np.array([1, 2, 3, 2, 5, 2])
print(np.where(a == 2))   # multiple occurrences

(array([2]),)
(array([1, 3, 5]),)


### Array sorting

We sort numpy arrays with the `sort()` function. Note that `np.sort(a)` returns a **new** sorted array, while the *method* `a.sort()` sorts **in place** and changes the original:

In [17]:
a = np.array([9, 5, 1, 3, 0, 2, -5, -8])
print(np.sort(a))
print(a)        # a itself is unchanged

[-8 -5  0  1  2  3  5  9]
[ 9  5  1  3  0  2 -5 -8]


In [18]:
a = np.array([9, 5, 1, 3, 0, 2, -5, -8])
a.sort()        # in place — a is modified
print(a)

[-8 -5  0  1  2  3  5  9]


### Filtering numpy arrays

For python lists, filtering requires a loop with an `if` inside:

In [19]:
a = [2, 5, 4, 8, 4, 1]
for i in a:
    if i < 4:
        print(i)

2
1


In numpy, the same filter is a single line. Writing `a < 4` creates a *boolean array* (an element-wise comparison — vectorisation again), and indexing with a boolean array keeps only the `True` positions:

In [20]:
a = np.array([2, 5, 4, 8, 4, 1])
print(a < 4)
print(a[a < 4])

[ True False False False False  True]
[2 1]


Boolean masking like this is one of the most-used idioms in data python — you will use it constantly with `pandas`.

## Data workhorse: `pandas`

`pandas` is a data-structure library particularly suited to **tabular** data, and it has become the global standard for representing and manipulating data in python.

`pandas` represents data as an object called a `DataFrame`. Think of it as a spreadsheet table: named columns, plus an *index* that labels the rows. (Unlike a database primary key, a pandas index does **not** have to be unique.) DataFrames are powerful because of their querying and transformation methods.

Let's create a simple `DataFrame` recording three fruits with their price and colour:

In [21]:
import pandas as pd
p = pd.DataFrame({'fruit': ['apple', 'banana', 'pear'],
                  'price': [4, 3.5, 2],
                  'color': ['red', 'yellow', 'green']})
print(p)

    fruit  price   color
0   apple    4.0     red
1  banana    3.5  yellow
2    pear    2.0   green


Printing a `DataFrame` formats it as a table, with the index (the untitled first column) and the titled columns.

We can retrieve a row by its index label using `loc`:

In [22]:
p.loc[1]

fruit    banana
price       3.5
color    yellow
Name: 1, dtype: object

It is tempting to read `loc[1]` as "the second row", but `loc` selects by **index label**, and labels and positions only coincide when the index is the default `0, 1, 2, ...`. When you genuinely need positional selection, use `iloc` (integer-location) instead:

In [23]:
print(p.loc[1])    # the row whose label is 1
print(p.iloc[1])   # the row at position 1 (same here, by coincidence)

fruit    banana
price       3.5
color    yellow
Name: 1, dtype: object
fruit    banana
price       3.5
color    yellow
Name: 1, dtype: object


The real power of `loc` is filtering with a boolean condition — the same masking idiom as in numpy. Here is the `banana` record:

In [24]:
p.loc[p.fruit == 'banana']

,fruit,price,color
1,banana,3.5,yellow


And all records with prices higher than 2:

In [25]:
p.loc[p.price > 2]

,fruit,price,color
0,apple,4.0,red
1,banana,3.5,yellow


## Loading and saving CSV files

One of the most useful `pandas` features is reading and writing CSV files. Let's save the fruit `DataFrame` as `fruits.csv`:

In [26]:
p.to_csv('fruits.csv')

And read it back. The first argument of `read_csv` is the file name; `index_col=0` tells pandas to use the first column as the index instead of creating a fresh one:

In [27]:
q = pd.read_csv('fruits.csv', index_col=0, header=0)
print(q)

    fruit  price   color
0   apple    4.0     red
1  banana    3.5  yellow
2    pear    2.0   green


### Finding missing values

Let's load a small dataset that contains gaps, `some_data.csv` (in this folder), and check for missing values.

In [28]:
q = pd.read_csv('some_data.csv', index_col='index', header=0)
print(q)

             A     B
index               
1          NaN  56.0
2         34.0  54.0
3        123.0  52.0
4          NaN   NaN
5      23432.0   NaN
6         23.0  76.0


The blank cells are labelled `NaN` ("not a number"). `pandas` finds them with the `isnull()` method, which returns a boolean `DataFrame` — `True` wherever a value is missing:

In [29]:
q.isnull()

,A,B
index,,
1,True,False
2,False,False
3,False,False
4,True,True
5,False,True
6,False,False


We can combine this with the masking idiom we learned above. `.isnull().any(axis=1)` returns `True` for every **row** that contains at least one `NaN` (`axis=1` means "collapse along columns, i.e. per row"); indexing with that boolean Series lists the offending row labels:

In [30]:
rows_with_missing = q.isnull().any(axis=1)
print(q.index[rows_with_missing])

Index([1, 4, 5], dtype='int64', name='index')


Note that `any(axis=1)` needs its argument as a **keyword**: `axis=1` selects the row-wise direction, `axis=0` (the default) would check per column.

### Imputation: filling `NaN` values

The simplest imputation replaces every missing value with a constant, using `fillna()`:

In [31]:
filled = q.fillna(3)
print(filled)

             A     B
index               
1          3.0  56.0
2         34.0  54.0
3        123.0  52.0
4          3.0   3.0
5      23432.0   3.0
6         23.0  76.0


Important: `fillna()` returns a **new** `DataFrame` — the original `q` is unchanged unless you assign the result back (`q = q.fillna(3)`) or pass `inplace=True`. This "methods return new objects" convention is everywhere in pandas and is a classic beginner trap.

A constant is rarely the best fill value. A more common choice is a column statistic such as the mean — but before filling gaps with statistics, we must deal with **outliers**, because extreme values corrupt those very statistics. That is the next topic.

## Detecting and removing outliers

There *is* an outlier hiding in `some_data.csv`: the value `23432`, roughly a thousand times larger than the other values in its column. Two standard detection rules are:

- **Z-score rule**: flag values more than (usually) 3 standard deviations from the mean.
- **IQR rule**: flag values below $Q_1 - 1.5\,\mathrm{IQR}$ or above $Q_3 + 1.5\,\mathrm{IQR}$, where $\mathrm{IQR} = Q_3 - Q_1$ is the interquartile range.

The IQR rule is less sensitive to the outlier itself (the mean and standard deviation are *pulled* by extreme values, quartiles much less so), so it is usually preferred. `pandas` computes quantiles directly (ignoring `NaN` by default):

In [32]:
q1 = q['A'].quantile(0.25)
q3 = q['A'].quantile(0.75)
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
print(f"Acceptable range: [{lower:.1f}, {upper:.1f}]")

outlier_rows = (q['A'] < lower) | (q['A'] > upper)
print(q.loc[outlier_rows])

q_clean = q.loc[~outlier_rows]
print(q_clean)

Acceptable range: [-8847.2, 14828.8]
             A   B
index             
5      23432.0 NaN
           A     B
index             
1        NaN  56.0
2       34.0  54.0
3      123.0  52.0
4        NaN   NaN
6       23.0  76.0


Note the pieces you already know: `|` combines boolean masks element-wise ("or"), and `~` negates a mask ("not"). Dropping the row is appropriate here because 23432 is plainly a data-entry error; milder outliers are sometimes *capped* at the boundary instead of removed.

## Imputation with statistics — mind the order

Now that column `A` is clean, filling its gaps with its mean is sensible:

In [33]:
q_filled = q_clean.copy()
for col in q_clean.columns:
    q_filled[col] = q_clean[col].fillna(q_clean[col].mean())
print(q_filled)

           A     B
index             
1       60.0  56.0
2       34.0  54.0
3      123.0  52.0
4       60.0  59.5
6       23.0  76.0


To see why the **order matters**, try imputing *before* removing the outlier: the mean of column `A` including `23432` comes out around `5903` — a value far from every genuine observation — and the gaps would be filled with exactly that. Garbage in, garbage out. The safe order is:

1. detect and handle outliers
2. impute missing values

Other standard imputation strategies are the column median (more robust to outliers), the mode (for categorical columns), or `sklearn.impute.SimpleImputer`, which you will meet in the case studies.

## One-hot encoding in practice

The fruit table has a categorical `color` column — exactly the situation from the "One-hot encoding" section above. Let's encode it numerically:

In [34]:
encoded = pd.get_dummies(p, columns=['color'], dtype=int)
print(encoded)

    fruit  price  color_green  color_red  color_yellow
0   apple    4.0            0          1             0
1  banana    3.5            0          0             1
2    pear    2.0            1          0             0


`get_dummies` replaced the single `color` column with one column **per category value**, containing a 1 where the row had that colour and 0 elsewhere — each fruit is now a vector of zeros and ones, just like the red/blue/green/black example earlier. Models fed these columns cannot invent an ordering between colours. (The `dtype=int` argument makes pandas use integers; without it, recent pandas versions produce boolean `True`/`False` columns, which encode the same information.)

## Text manipulation in python

One of the factors that catalysed python's popularity is its versatility in text manipulation. The most-used string methods are:

- `s.upper()` and `s.lower()`
- `s.strip()` — remove whitespace (or chosen characters) from both ends
- `s.split()` — split a string into a list of words
- `s.replace(a, b)` — replace all occurrences of `a` with `b`
- `sep.join(list)` — join a list of strings into one, with `sep` between them
- `s.isnumeric()` — check whether the string consists of digits only

In [35]:
a = 'this is some text'
print(a.upper(), a.lower())

a = '   lots of trailing spaces.    '
print(a.strip())

print(a.strip().split())

a = 'abcddefdd'
print(a.replace('d', 'x'))

print('abc'.join(['x', 'y', 'z']))

a = '456a'
print(a.isnumeric())

THIS IS SOME TEXT this is some text
lots of trailing spaces.
['lots', 'of', 'trailing', 'spaces.']
abcxxefxx
xabcyabcz
False


Let's apply them to the news sentence from the introduction. The preprocessing recipe discussed there was: lower-case everything, then strip punctuation:

In [36]:
sentence = ("Foreign Affairs Minister Penny Wong has asked Optus to cover "
            "passport application fees for anyone caught up in last week's "
            "massive data breach, which affected millions of Australians.")

clean = sentence.lower().replace(',', '').replace('.', '').split()
print(clean[:10], '...')

['foreign', 'affairs', 'minister', 'penny', 'wong', 'has', 'asked', 'optus', 'to', 'cover'] ...


The gap between this and "real" natural-language processing — handling "has"/"have" as one token, recognising that Penny Wong is a person — is filled by techniques like *stemming*, *lemmatisation* and *token vectors*, which you will see in Case Study 06.

These string methods are used constantly when pre-processing text for natural language processing — that is the *text cleaning* step.

That's all for this class. Next up: [exploratory data analysis](../Class03_ExploratoryDataAnalysis/EDA.ipynb), where we use these tools to interrogate a real, messy dataset. You can practise this class's skills in `Lab_DataEngineering.ipynb` (with a solutions notebook alongside).